## Cél

Ez a notebook a `quant_train` tábla feature-jeinek szisztematikus minőségellenőrzését végzi solusdt, 1m bontásban.
A 4 elemzési lépés sorban fut: **quality → target_relation → redundancy → stability**.
Az eredmény determinisztikusan kerül a `feature_set.json`-be — nincs manuális szerkesztés.

**Kapcsolódó dokumentáció:**
- Módszertan: `_doc_/2000_features.md`
- Kód-dokumentáció: `_doc_/2010_feature_engineering.md`

**Asset:** SOLUSDT | **Granularitás:** 1m | **Dátum:** 2026-06-19

In [ ]:
import sys
import json
import logging
from pathlib import Path
from datetime import datetime

import duckdb
import pandas as pd
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from IPython.display import display, Markdown

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(_root / "src"))

import utils
from analyst.table_formatting import display_analysis_table
from modeling.feature_engineering import (
    FeatureEngineeringConfig,
    analyze_quality,
    analyze_target_relation,
    analyze_redundancy,
    analyze_stability,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)

CQ_COLORS = {
    "blue":       "#1696d2",
    "black":      "#000000",
    "gray_dark":  "#353535",
    "gray":       "#696969",
    "gray_light": "#d2d2d2",
    "yellow":     "#fdbf11",
    "orange":     "#f15a24",
    "red":        "#ec008b",
    "green":      "#55b748",
}
CQ_SEQUENCE = [
    CQ_COLORS["blue"], CQ_COLORS["yellow"], CQ_COLORS["orange"],
    CQ_COLORS["gray"], CQ_COLORS["red"],
]
DECISION_COLORS = {
    "keep":   CQ_COLORS["green"],
    "review": CQ_COLORS["yellow"],
    "drop":   CQ_COLORS["red"],
}

sns.set_theme(
    style="whitegrid",
    rc={
        "figure.figsize":    (9, 5),
        "figure.dpi":        120,
        "axes.spines.top":   False,
        "axes.spines.right": False,
        "axes.edgecolor":    CQ_COLORS["gray"],
        "axes.labelcolor":   CQ_COLORS["gray_dark"],
        "xtick.color":       CQ_COLORS["gray_dark"],
        "ytick.color":       CQ_COLORS["gray_dark"],
        "grid.color":        CQ_COLORS["gray_light"],
        "grid.linewidth":    0.8,
        "axes.axisbelow":    True,
        "legend.frameon":    False,
    },
)
sns.set_palette(CQ_SEQUENCE)

In [ ]:
ASSET_ID = "solusdt"
RUN_ID   = datetime.now().strftime("run_%Y%m%d_%H%M%S")

asset_cfg  = utils.load_asset_config(ASSET_ID)
db_path    = asset_cfg["database"]["db_path"]
data_dir   = Path(db_path).parent
output_dir = data_dir / "feature_engineering" / RUN_ID
cfg        = FeatureEngineeringConfig(asset_id=ASSET_ID, run_id=RUN_ID)

conn   = duckdb.connect(db_path, read_only=True)
n_rows = conn.execute("SELECT COUNT(*) FROM quant_train").fetchone()[0]

display(Markdown(f"""
| Paraméter | Érték |
|-----------|-------|
| asset_id  | `{ASSET_ID}` |
| run_id    | `{RUN_ID}` |
| db        | `{db_path}` |
| output    | `{output_dir}` |
| sorok     | **{n_rows:,}** |
"""))

## Quality — Univariáns minőség szűrés

**Mi ez.** Minden `feat_*` oszlopra kiszámítjuk a null arányt, az inf értékek arányát, a varianciát és az outlier hányadot (|z-score| > 3).

**Forrás.** `quant_train` tábla, összes `feat_*` oszlop.

**Értelmezés.** `drop` = null_rate > 0.01 VAGY inf_rate > 0.001 VAGY variance < 1e-8; `review` = outlier_ratio > 0.05 (és nincs drop ok); `keep` = minden más. Az első 1441 sor null az t-1 lag warmup miatt — ez nem minőségi hiba, a sampling garantálja, hogy ezek nem kerülnek a tanítási ablakba.

In [ ]:
#| label: tbl-quality-decisions
#| tbl-cap: "Feature quality döntések — összes feature"

quality_df = analyze_quality(conn, cfg)
display_analysis_table(quality_df.to_pandas())

In [ ]:
#| label: fig-quality-decisions
#| fig-cap: "Quality döntések megoszlása"
#| fig-alt: "Oszlopdiagram: keep/review/drop döntések száma"

vc = quality_df["decision"].value_counts().sort("decision")
labels = vc["decision"].to_list()
counts = vc["count"].to_list()

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(labels, counts, color=[DECISION_COLORS.get(l, CQ_COLORS["gray"]) for l in labels])
ax.set_xlabel("Döntés")
ax.set_ylabel("Feature count")
for i, v in enumerate(counts):
    ax.text(i, v + 0.3, str(v), ha="center", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
#| label: tbl-quality-nokeep
#| tbl-cap: "Nem-keep quality döntések részletei"

not_keep_q = quality_df.filter(pl.col("decision") != "keep").select(
    ["feature", "null_rate", "inf_rate", "variance", "outlier_ratio", "decision", "drop_reason"]
).sort("decision")

display(Markdown(f"**Nem-keep features:** {len(not_keep_q)}"))
display_analysis_table(not_keep_q.to_pandas())

## Target Relation — Szignálerősség

**Mi ez.** Minden `feat_*` × target párra Pearson és Spearman korrelációt számítunk a `quant_train` táblán. `signal_proxy = |ρ_spearman|`.

**Forrás.** `quant_train` tábla, `feat_*` és `long_mfe_fw60`, `short_mfe_fw60` oszlopok.

**Értelmezés.** `leakage` = |ρ| > 0.95 (gyanús); `weak` = |ρ| < 0.01 (nincs szignál); `keep` = minden más. Egy feature akkor kerül ki, ha **mindkét** targetre `weak` (vagy `leakage`).

In [ ]:
#| label: tbl-relation-head
#| tbl-cap: "Target relation — top 10 sor (signal_proxy szerint rendezve)"

relation_df = analyze_target_relation(conn, cfg)
display_analysis_table(
    relation_df.sort("signal_proxy", descending=True).head(10).to_pandas()
)

In [ ]:
#| label: fig-relation-top20
#| fig-cap: "Top 20 feature signal_proxy értéke targetenként"
#| fig-alt: "Vízszintes oszlopdiagram: top 20 feature Spearman korrelációja long és short targettel"
#| fig-subcap:
#|   - "long_mfe_fw60"
#|   - "short_mfe_fw60"
#| layout-ncol: 1

for target in cfg.target_cols:
    sub = (
        relation_df
        .filter(pl.col("target") == target)
        .sort("signal_proxy", descending=True)
        .head(20)
    ).to_pandas()

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(sub["feature"], sub["signal_proxy"], color=CQ_COLORS["blue"])
    ax.invert_yaxis()
    ax.set_xlabel("|ρ_spearman|")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

In [ ]:
#| label: tbl-relation-nokeep
#| tbl-cap: "Nem-keep (feature, target) párok"

not_keep_r = relation_df.filter(pl.col("decision") != "keep").sort(["feature", "target"])
display(Markdown(f"**Nem-keep (feature, target) párok:** {len(not_keep_r)}"))
display_analysis_table(not_keep_r.to_pandas())

## Redundancy — Korrelációs klaszterezés

**Mi ez.** A `feat_*` oszlopokat Pearson korrelációs mátrix alapján klaszterezzük (union-find). Ha két feature |Pearson r| ≥ 0.95, egy klaszterbe kerülnek. Klaszterenként egy reprezentatív marad (`keep`), a többi `drop`.

**Forrás.** `quant_train` tábla, 500 000 véletlen sorból számított korrelációs mátrix.

**Értelmezés.** Reprezentatív = legkisebb indexű feature a klaszterben. A downstream `02_hyper_param_search.py` LightGBM feature importance-on keresztül tovább szűrhet.

In [ ]:
#| label: tbl-redundancy-head
#| tbl-cap: "Redundancy analízis — top 10 sor"

redundancy_df = analyze_redundancy(conn, cfg)
display_analysis_table(redundancy_df.head(10).to_pandas())

In [ ]:
#| label: tbl-redundancy-dropped
#| tbl-cap: "Redundáns (drop) feature-ök klaszter szerinti bontásban"

n_clusters = redundancy_df["cluster_id"].n_unique()
n_multi    = (
    redundancy_df.group_by("cluster_id")
    .agg(pl.len().alias("n"))
    .filter(pl.col("n") > 1)
    .height
)
n_dropped  = (redundancy_df["decision"] == "drop").sum()

display(Markdown(
    f"**Klaszterek:** {n_clusters} összesen, {n_multi} darab >1 taggal — "
    f"**{n_dropped} redundáns feature kizárva.**"
))

dropped_red = redundancy_df.filter(pl.col("decision") == "drop").select(
    ["feature", "cluster_id", "max_pearson", "drop_reason"]
).sort("cluster_id")
display_analysis_table(dropped_red.to_pandas())

## Stability — Időbeli stabilitás

**Mi ez.** Az adatot 90 napos, nem-átfedő időablakokra osztjuk. Minden (feature, bucket) párra Spearman korrelációt számítunk mindkét targettel és összehasonlítjuk a globális baseline-nal. `drift = |ρ_bucket − ρ_baseline|`.

**Forrás.** `quant_train` tábla, összes `feat_*` oszlop, 90 napos bucket-ek.

**Értelmezés.** `stable` ≤ 0.15; `review` 0.15–0.30; `unstable` > 0.30 (nem utolsó 2 bucket); `decayed` > 0.30 **és** utolsó 2 bucket → kizárjuk. A `decayed` flag azt jelenti, hogy a feature a legutóbbi adatszakaszban elveszítette prediktív kapcsolatát.

In [ ]:
#| label: tbl-stability-head
#| tbl-cap: "Stability analízis — top 10 sor"

stability_df = analyze_stability(conn, cfg)
display(Markdown(f"**Összes (feature, bucket) sor:** {len(stability_df):,}"))
display_analysis_table(stability_df.head(10).to_pandas())

In [ ]:
#| label: fig-stability-drift
#| fig-cap: "Drift időbeli lefutása — decayed/unstable feature-ök mintája (max 6)"
#| fig-alt: "Vonaldiagram: drift értékek bucket-enként, long és short targetekre"
#| layout-ncol: 1

problematic = (
    stability_df
    .filter(pl.col("stability_flag").is_in(["decayed", "unstable"]))
    ["feature"].unique().to_list()
)

if not problematic:
    display(Markdown("**Nincs decayed/unstable feature** — minden feature stabilan tartja a prediktív kapcsolatát."))
else:
    sample_feats = sorted(problematic)[:6]
    for feat in sample_feats:
        sub = stability_df.filter(pl.col("feature") == feat).sort("bucket_idx").to_pandas()
        fig, ax = plt.subplots(figsize=(10, 3))
        ax.plot(sub["bucket_idx"], sub["drift_long"],  marker="o", label="drift_long",  color=CQ_COLORS["blue"])
        ax.plot(sub["bucket_idx"], sub["drift_short"], marker="s", label="drift_short", color=CQ_COLORS["orange"])
        ax.axhline(cfg.max_drift_threshold, color=CQ_COLORS["red"], linestyle="--",
                   linewidth=1, label=f"threshold ({cfg.max_drift_threshold})")
        ax.legend(fontsize=8)
        ax.set_xlabel("bucket_idx")
        ax.set_ylabel("drift")
        ax.set_xlabel(f"{feat} — bucket_idx")
        plt.tight_layout()
        plt.show()

In [ ]:
#| label: tbl-stability-decayed
#| tbl-cap: "Decayed feature-ök összesítő statisztikái"

decayed_features = (
    stability_df
    .filter(pl.col("stability_flag") == "decayed")
    ["feature"].unique().to_list()
)
display(Markdown(f"**Decayed (kizárt) features:** {len(decayed_features)}"))

if decayed_features:
    decayed_summary = (
        stability_df
        .filter(pl.col("feature").is_in(decayed_features))
        .group_by("feature")
        .agg([
            pl.col("drift_long").max().alias("max_drift_long"),
            pl.col("drift_short").max().alias("max_drift_short"),
            pl.col("stability_flag")
              .filter(pl.col("stability_flag") == "decayed")
              .len().alias("decayed_buckets"),
        ])
        .sort("max_drift_long", descending=True)
    )
    display_analysis_table(decayed_summary.to_pandas())

## Output — feature_set.json

**Mi ez.** A 4 lépés eredményét kombináljuk. Egy feature a `selected` listába kerül, ha: (1) quality → `keep`, (2) target_relation → legalább egy targetre `keep`, (3) redundancy → `keep`, (4) stability → nincs `decayed` bucket.

**Forrás.** A fenti 4 analízis lépés kimenetei.

**Értelmezés.** A `selected` lista adódik tovább a `00_create_sample.py`-nak. `dropped` okkal, `review` marginális (megtartjuk, de jelölt).

In [ ]:
conn.close()
output_dir.mkdir(parents=True, exist_ok=True)

all_features = quality_df["feature"].to_list()
created_at   = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

quality_drop   = {r["feature"]: r["drop_reason"] for r in quality_df.iter_rows(named=True) if r["decision"] == "drop"}
quality_review = {r["feature"] for r in quality_df.iter_rows(named=True) if r["decision"] == "review"}

relation_keep = {r["feature"] for r in relation_df.iter_rows(named=True) if r["decision"] == "keep"}
relation_leak = {r["feature"] for r in relation_df.iter_rows(named=True) if r["decision"] == "leakage"}

redundancy_drop = {r["feature"]: r["drop_reason"] for r in redundancy_df.iter_rows(named=True) if r["decision"] == "drop"}

decayed_set  = {r["feature"] for r in stability_df.iter_rows(named=True) if r["stability_flag"] == "decayed"}
unstable_set = {r["feature"] for r in stability_df.iter_rows(named=True) if r["stability_flag"] in ("unstable", "review")} - decayed_set

selected: list[str]  = []
dropped:  list[dict] = []
review:   list[str]  = []

for feat in all_features:
    reasons: list[str] = []
    if feat in quality_drop:
        reasons.append(f"quality: {quality_drop[feat]}")
    if feat in relation_leak:
        reasons.append("relation: leakage suspect")
    if feat not in relation_keep and feat not in relation_leak:
        reasons.append("relation: weak signal across all targets")
    if feat in redundancy_drop:
        reasons.append(f"redundancy: {redundancy_drop[feat]}")
    if feat in decayed_set:
        reasons.append("stability: decayed in recent buckets")

    is_review = (feat in quality_review or feat in unstable_set) and not reasons

    if reasons:
        dropped.append({"col": feat, "reason": " | ".join(reasons)})
    elif is_review:
        review.append(feat)
    else:
        selected.append(feat)

fs = {
    "run_id"     : cfg.run_id,
    "asset_id"   : cfg.asset_id,
    "created_at" : created_at,
    "target_cols": list(cfg.target_cols),
    "selected"   : selected,
    "dropped"    : dropped,
    "review"     : review,
    "thresholds" : {
        "max_null_rate"        : cfg.max_null_rate,
        "max_inf_rate"         : cfg.max_inf_rate,
        "min_variance"         : cfg.min_variance,
        "max_outlier_ratio"    : cfg.max_outlier_ratio,
        "min_spearman_abs"     : cfg.min_spearman_abs,
        "max_spearman_leakage" : cfg.max_spearman_leakage,
        "pearson_cluster_thr"  : cfg.pearson_cluster_thr,
        "redundancy_max_rows"  : cfg.redundancy_max_rows,
        "stability_bucket_days": cfg.stability_bucket_days,
        "max_drift_threshold"  : cfg.max_drift_threshold,
    },
}

json_path = output_dir / "feature_set.json"
json_path.write_text(json.dumps(fs, indent=2), encoding="utf-8")

In [ ]:
#| label: tbl-output-summary
#| tbl-cap: "Feature szelekció végeredmény"

summary_df = pd.DataFrame([
    {"kategória": "selected", "count": len(selected)},
    {"kategória": "review",   "count": len(review)},
    {"kategória": "dropped",  "count": len(dropped)},
])
display_analysis_table(summary_df)

display(Markdown(
    f"**Output:** `{output_dir}` → `feature_set.json` ✓"
))

In [ ]:
#| label: fig-output-summary
#| fig-cap: "Feature szelekció végeredmény — megoszlás"
#| fig-alt: "Oszlopdiagram: selected/review/dropped feature count"

fig, ax = plt.subplots(figsize=(5, 3))
cats   = ["selected", "review", "dropped"]
values = [len(fs["selected"]), len(fs["review"]), len(fs["dropped"])]
colors = [CQ_COLORS["green"], CQ_COLORS["yellow"], CQ_COLORS["red"]]
ax.bar(cats, values, color=colors)
ax.set_xlabel("Kategória")
ax.set_ylabel("Feature count")
for i, v in enumerate(values):
    ax.text(i, v + 0.3, str(v), ha="center", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()